# Session Browser 구축하기

Agent를 제품(desktop app, IDE extension, 내부 chatbot)으로 출시하면 사용자가 가장 먼저 요청하는 것은 sidebar입니다. 지난 화요일의 대화를 보고 다시 이어가거나, 원본을 잃지 않고 새 방향으로 분기하고 싶어 합니다. Agent loop가 제품의 절반이라면, 나머지 절반은 session 관리입니다.

Claude Agent SDK는 모든 대화를 disk의 JSONL transcript에 기록합니다. Transcript를 다시 읽고 정리하는 function도 제공하므로, file parser를 작성하거나 `~/.claude/projects/`를 직접 탐색하지 않고 sidebar를 구축할 수 있습니다.

**이 cookbook을 마치면 다음을 할 수 있습니다:**

- Pagination과 branch, title, 마지막 수정 시간 같은 metadata를 포함해 프로젝트의 과거 session 목록을 조회하고 rendering하기
- Agent를 시작하지 않고 저장된 session의 message를 UI로 다시 읽기
- 사용자가 기록을 정리할 수 있도록 session 이름 변경, tag 지정, filtering하기
- 어느 지점에서든 session을 fork하고 실제 `query()` 호출로 fork를 resume하기

이것이 Claude Code Desktop과 VS Code extension의 session sidebar에 사용되는 pattern입니다. Agent SDK 위에 어떤 UI를 구축하든 같은 primitive를 사용할 수 있습니다.

## 사전 요구사항

이 가이드를 따라 하기 전에 다음을 준비하세요:

**필수 지식**

- `async`/`await`를 포함한 Python 기초
- Agent SDK의 `query()` function에 대한 기본 이해(소개는 [Notebook 00](00_The_one_liner_research_agent.ipynb) 참고)

**필수 Tool**

- Python 3.11 이상
- Claude Code CLI(`npm install -g @anthropic-ai/claude-code`)
- Anthropic API key([여기에서 발급](https://console.anthropic.com))

## 설정

필요한 dependency를 설치합니다. Session 관리 function은 `claude-agent-sdk` v0.1.51에 추가되었습니다.

In [ ]:
%%capture
%pip install -U "claude-agent-sdk>=0.1.51" python-dotenv pandas

`.env`에서 API key를 load하고 model을 구성합니다. Demo session이 짧으며 저렴하고 빠르기를 원하므로 여기서는 Haiku를 사용합니다. 실제 제품에서는 agent에 맞는 model을 선택하면 됩니다.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

MODEL = "claude-haiku-4-5"

# 모든 demo session은 이 프로젝트 디렉터리에 있습니다. 전용
# cwd를 사용하면 실제 Claude Code session과 demo를 격리합니다.
# 참고: 이 경로는 kernel의 작업 디렉터리를 기준으로 해석됩니다
# (README에 따라 실행하면 claude_agent_sdk/).
DEMO_DIR = str(Path("session_browser_demo").resolve())
os.makedirs(DEMO_DIR, exist_ok=True)
print(f"Demo project dir: {DEMO_DIR}")

# 1부: 관리할 Session 생성하기

Session 관리 function은 `query()`(또는 Claude Code CLI)가 이미 작성한 transcript를 읽습니다. 탐색할 대화를 확보하기 위해 먼저 짧은 대화 세 개를 실행하고 session ID를 가져옵니다.

`cwd` 옵션은 이 대화가 어느 프로젝트 디렉터리에 속하는지 SDK에 알려 줍니다. Transcript는 `~/.claude/projects/<encoded-cwd>/<session-id>.jsonl`에 저장되므로 같은 `cwd`의 모든 호출은 같은 bucket으로 들어갑니다. 이후 `list_sessions(directory=...)`가 읽는 것이 이 bucket입니다.

Token 사용을 최소화하기 위해 tool을 비활성화하고 각 실행을 한 turn으로 제한합니다.

In [ ]:
from claude_agent_sdk import ClaudeAgentOptions, ResultMessage, query


async def run_one_turn(prompt: str) -> str:
    """Single-turn 대화를 실행하고 session_id를 반환합니다."""
    opts = ClaudeAgentOptions(
        model=MODEL,
        cwd=DEMO_DIR,
        max_turns=1,
        allowed_tools=[],  # Text만 사용, tool loop 없음
    )
    session_id = None
    async for msg in query(prompt=prompt, options=opts):
        if isinstance(msg, ResultMessage):
            session_id = msg.session_id
            preview = (msg.result or "")[:80]
            print(f"[{session_id[:8]}] {preview}...")
    if session_id is None:
        raise RuntimeError("No ResultMessage received; check API key and SDK version.")
    return session_id

Prompt 세 개, session 세 개입니다. 실제 제품에서는 사용자가 agent에 요청한 내용이 될 것입니다.

In [ ]:
prompts = [
    "Give me three name ideas for a CLI tool that manages git worktrees.",
    "Explain the difference between a mutex and a semaphore in one paragraph.",
    "Write a haiku about merge conflicts.",
]

demo_session_ids = []
for p in prompts:
    sid = await run_one_turn(p)
    demo_session_ids.append(sid)

print(f"\nCreated {len(demo_session_ids)} sessions.")

# 2부: Session 목록 조회 및 점검

## Session 목록 구축하기

`list_sessions()`는 프로젝트의 transcript 디렉터리를 scan하여 각 session의 metadata를 최신순으로 반환합니다. 파일 정보와 각 transcript의 앞부분 및 뒷부분을 읽으므로 디렉터리에 수백 개의 파일이 있어도 빠릅니다. Subprocess를 시작하지도 API를 호출하지도 않습니다.

각 `SDKSessionInfo`에는 picker의 행을 rendering하는 데 필요한 표시용 요약, 마지막 수정 timestamp, git branch, 작업 디렉터리, 설정한 custom title이나 tag가 담깁니다.

In [ ]:
from datetime import datetime

import pandas as pd

from claude_agent_sdk import SDKSessionInfo, list_sessions

sessions = list_sessions(directory=DEMO_DIR)

# Table로 rendering. 실제 UI에서는 sidebar 구성 요소입니다.
rows = []
for s in sessions:
    rows.append(
        {
            "id": s.session_id[:8],
            "summary": (s.summary[:50] + "...") if len(s.summary) > 50 else s.summary,
            "modified": datetime.fromtimestamp(s.last_modified / 1000).strftime("%H:%M:%S"),
            "branch": s.git_branch or "-",
            "tag": s.tag or "-",
        }
    )

pd.DataFrame(rows)

기록이 길면 `limit`와 `offset`으로 결과를 page 단위로 조회하세요. Session picker는 보통 첫 page를 load한 뒤 사용자가 scroll하면 추가로 가져옵니다:

```python
page_2 = list_sessions(directory=DEMO_DIR, limit=20, offset=20)
```

App이 이미 session ID를 저장했다면(예: 자체 database의 사용자 기록과 함께 저장) 해당 행 하나만 가져올 때는 전체 목록을 조회하는 것보다 `get_session_info()`가 더 적은 비용이 듭니다.

In [ ]:
from claude_agent_sdk import get_session_info

info = get_session_info(demo_session_ids[0], directory=DEMO_DIR)

print(f"Session:      {info.session_id}")
print(f"Summary:      {info.summary}")
print(f"First prompt: {info.first_prompt}")
print(f"Created:      {datetime.fromtimestamp(info.created_at / 1000)}")
print(f"Size:         {info.file_size:,} bytes")

## Session의 Message 읽기

사용자가 sidebar에서 session을 클릭하면 main view로 대화를 load합니다. `get_session_messages()`는 transcript에서 message chain을 재구성하고 user 및 assistant turn을 순서대로 반환합니다. 목록 조회 function과 마찬가지로 순수한 파일 읽기이므로 agent가 실행 중일 필요가 없습니다.

각 `SessionMessage`에는 `type`(`"user"` 또는 `"assistant"`), `uuid`, Anthropic Messages API와 같은 형태(`role`, `content`)의 `message` dict가 있습니다.

In [ ]:
from claude_agent_sdk import get_session_messages

messages = get_session_messages(demo_session_ids[0], directory=DEMO_DIR)

for m in messages:
    role = m.type
    # content는 block 목록이며 text block을 추출함
    text_parts = [
        b.get("text", "")
        for b in m.message.get("content", [])
        if isinstance(b, dict) and b.get("type") == "text"
    ]
    text = " ".join(text_parts).strip()
    print(f"[{role:>9}] {text[:100]}")

긴 session에서는 `limit`와 `offset`으로 한 구간씩 load할 수 있습니다. Chat view는 열릴 때 마지막 message 50개를 load하고 사용자가 위로 scroll하면 이전 page를 가져올 수 있습니다. Offset은 시간순(오래된 것부터)으로 적용되므로 page 0은 대화의 시작입니다.

# 3부: Title 및 Tag로 정리하기

## Session 이름 변경하기

자동 생성된 요약은 빠르게 훑어보기에는 좋지만, 사용자는 session에 실제 이름을 붙이고 싶어 하는 경우가 많습니다. `rename_session()`은 transcript에 title 항목을 append하며, `list_sessions()`는 다음 읽기에서 이를 `custom_title`로 가져옵니다.

Append는 비용이 적고 idempotent합니다. 이름 변경을 두 번 호출하면 더 최근의 title이 적용될 뿐입니다. 파일을 다시 쓰지는 않습니다.

In [ ]:
from claude_agent_sdk import rename_session

rename_session(demo_session_ids[0], "Worktree CLI naming brainstorm", directory=DEMO_DIR)
rename_session(demo_session_ids[2], "Haiku corner", directory=DEMO_DIR)

# Title이 유지되는지 확인
for s in list_sessions(directory=DEMO_DIR):
    label = s.custom_title or "(auto)"
    print(f"{s.session_id[:8]}  custom_title={label!r}  summary={s.summary[:40]!r}")

## Tag 지정 및 Filtering

Tag는 session에 연결되는 하나의 문자열입니다. `"archived"`, `"needs-review"`, `"favorite"`처럼 제품에 필요한 분류에 사용하세요. Tag를 지우려면 `None`을 전달합니다.

흔한 pattern은 soft-delete입니다. Transcript 파일을 삭제하는 대신 `"__hidden"` tag를 붙이고 목록 view에서 filtering하여 제외합니다. Data는 복구 가능하게 남습니다.

In [ ]:
from claude_agent_sdk import tag_session

# 두 session을 favorite으로 표시하고 나머지는 숨김
tag_session(demo_session_ids[0], "favorite", directory=DEMO_DIR)
tag_session(demo_session_ids[2], "favorite", directory=DEMO_DIR)
tag_session(demo_session_ids[1], "__hidden", directory=DEMO_DIR)


def visible_sessions(directory: str, tag_filter: str | None = None) -> list[SDKSessionInfo]:
    """Soft-delete를 숨기고 선택적으로 tag로 filtering하여 session 목록을 조회합니다."""
    results = []
    for s in list_sessions(directory=directory):
        if s.tag == "__hidden":
            continue
        if tag_filter is not None and s.tag != tag_filter:
            continue
        results.append(s)
    return results


favorites = visible_sessions(DEMO_DIR, tag_filter="favorite")
print(f"Visible favorites: {len(favorites)}")
for s in favorites:
    print(f"  {s.session_id[:8]}  [{s.tag}]  {s.custom_title or s.summary}")

Tag는 목록이 아닌 단일 값입니다. Status와 category처럼 여러 축이 필요하면 `"review:urgent"` 같은 하나의 문자열에 encoding하고 읽을 때 parsing하거나, `session_id`를 key로 자체 database에 더 풍부한 state를 저장하세요.

# 4부: Fork 및 Resume

## 기존 대화에서 분기하기

Fork는 message ID를 새로 remapping하여 session의 transcript를 새 파일에 복사합니다. 원본은 그대로 유지됩니다. 이는 "다른 접근 방식 시도" 기능의 기반 primitive입니다. 사용자는 원래 thread를 유지하며 실험할 새 thread를 얻습니다.

`fork_session()`은 새 파일을 작성하고 ID를 반환합니다. Agent는 실행하지 않으므로, `query()`로 resume할 때까지 fork는 disk에만 존재합니다.

In [ ]:
from claude_agent_sdk import fork_session

source = demo_session_ids[0]

fork = fork_session(
    source,
    directory=DEMO_DIR,
    title="Worktree CLI names (round 2)",
)

print(f"Source: {source}")
print(f"Fork:   {fork.session_id}")

# Fork는 source와 같은 message 기록으로 시작함
source_msgs = get_session_messages(source, directory=DEMO_DIR)
fork_msgs = get_session_messages(fork.session_id, directory=DEMO_DIR)
print(f"Source has {len(source_msgs)} messages, fork has {len(fork_msgs)}")

전체 기록 대신 특정 지점에서 분기하려면 `up_to_message_id`를 전달하세요. Fork에는 해당 message까지 포함한 원본 transcript가 담깁니다. Message UUID는 `get_session_messages()[i].uuid`로 얻을 수 있습니다.

## Fork를 실제 Query로 Resume하기

Fork는 transcript 파일일 뿐입니다. 다시 실행 중인 대화로 만들려면 ID를 `ClaudeAgentOptions.resume`에 전달합니다. Agent는 fork된 기록을 load하고 그 지점부터 이어갑니다.

In [ ]:
resume_opts = ClaudeAgentOptions(
    model=MODEL,
    cwd=DEMO_DIR,
    max_turns=1,
    allowed_tools=[],
    resume=fork.session_id,
)

async for msg in query(
    prompt="Those were okay. Give me three more names, but punnier.",
    options=resume_opts,
):
    if isinstance(msg, ResultMessage):
        print(f"[fork {fork.session_id[:8]} resumed]")
        print(msg.result)

원래 session은 여전히 그대로 남아 있습니다. 다시 목록을 조회하면 원본과 fork가 독립적인 기록을 가진 별도 행으로 표시됩니다.

In [ ]:
for s in list_sessions(directory=DEMO_DIR):
    marker = "(fork)" if s.session_id == fork.session_id else "      "
    print(f"{marker} {s.session_id[:8]}  {s.custom_title or s.summary[:50]}")

# 정리

`delete_session()`은 transcript 파일을 삭제합니다. Hard delete이므로, 사용자용 UI에서는 보통 3부의 soft-delete tag pattern이 더 안전한 기본 선택입니다.

여기서는 demo에서 생성한 모든 것을 정리하는 데 사용합니다.

In [ ]:
from claude_agent_sdk import delete_session

# Fork를 포함한 demo 디렉터리의 모든 session 정리
for s in list_sessions(directory=DEMO_DIR):
    delete_session(s.session_id, directory=DEMO_DIR)
    print(f"Deleted {s.session_id[:8]}")

remaining = list_sessions(directory=DEMO_DIR)
print(f"\n{len(remaining)} session(s) remaining.")

# 되짚어 보기

Agent SDK의 로컬 transcript store를 기반으로 session browser의 핵심을 구축했습니다:

- `list_sessions()`를 사용한 **목록 조회**는 sidebar rendering에 필요한 모든 정보를 제공합니다. 전체 transcript를 parsing하는 대신 파일 정보 및 앞/뒤 일부만 읽으므로 확장에도 적합합니다.
- `get_session_messages()`를 사용한 **읽기**는 agent를 시작하지 않고 표시할 대화를 다시 load합니다.
- `rename_session()` 및 `tag_session()`을 사용한 **정리**는 metadata 항목을 append하므로 비용이 적으며 가장 최근 호출이 적용됩니다.
- `fork_session()`과 `options.resume`을 사용하는 **fork**는 사용자가 원본을 건드리지 않고 대화를 분기해 이어갈 수 있게 합니다.

모두 `~/.claude/projects/`에 대한 순수한 파일 작업입니다. Agent subprocess의 실행 여부와 관계없이 작동하며, Claude Code CLI가 작성하는 것과 동일한 transcript를 봅니다.

## 다음으로 할 일

- **UI에 연결하기.** 이 function들은 UI에 의존하지 않습니다. FastAPI route나 Electron IPC handler 뒤에 두면 session sidebar의 backend가 됩니다.
- **Host 간 Session.** Transcript는 로컬 disk에 있습니다. 컴퓨터 간에 session을 공유하려면 파일을 sync하거나 session ID 및 message를 자체 store에 indexing하세요. Pattern은 [Manage sessions on disk](https://docs.claude.com/en/agent-sdk/local-session-management)를 참고하세요.
- **TypeScript.** `@anthropic-ai/claude-agent-sdk`에도 camelCase 이름(`listSessions`, `forkSession` 등)으로 같은 API가 있습니다. [TypeScript SDK reference](https://docs.claude.com/en/agent-sdk/typescript)를 참고하세요.
- **전체 맥락.** Notebook [00](00_The_one_liner_research_agent.ipynb)부터 [03](03_The_site_reliability_agent.ipynb)까지는 agent 자체의 구축을 다룹니다. 이 Notebook은 agent가 남기는 것을 관리하는 방법을 다룹니다.